In [ ]:
import requests
import pandas as pd
import time

years_to_fetch = [2026] 
all_stations = []
overpass_url = "http://overpass-api.de/api/interpreter"

headers = {
    'User-Agent': 'UKCrimeDataProject/1.0 (Contact: catherinekasia@gmail.com)'
}

uk_regions = {
    "Scotland": "54.6,-8.0,60.9,0.0",
    "North England": "53.0,-4.0,54.6,0.0",
    "Wales & West Midlands": "51.4,-5.5,53.0,-1.0",
    "East Midlands & East Anglia": "51.4,-1.0,53.0,2.0",
    "South England": "49.9,-6.0,51.4,2.0",
    "Northern Ireland": "54.0,-8.5,55.3,-5.0"
}

for year in years_to_fetch:
    for region_name, bbox in uk_regions.items():
        print(f" on: {region_name}")
        
        #construct the Overpass QL query for police stations in the specified region and year
        overpass_query = f"""
        [out:json][timeout:180][date:"{year}-01-01T00:00:00Z"][bbox:{bbox}];
        (
          node["amenity"="police"];
          way["amenity"="police"];
          relation["amenity"="police"];
        );
        out center;
        """
        
        try:
            response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
            response.raise_for_status() 
            data = response.json()
            
            for element in data.get('elements', []):
                lat = element.get('lat', element.get('center', {}).get('lat'))
                lon = element.get('lon', element.get('center', {}).get('lon'))
                
                if lat and lon:
                    all_stations.append({
                        'year': year,
                        'latitude': lat,
                        'longitude': lon,
                        'name': element.get('tags', {}).get('name', 'Unknown Station')
                    })
                    
            time.sleep(8) 
            
        except Exception as e:
            print(f"  failed to fetch {region_name}: {e}")

stations_df = pd.DataFrame(all_stations)
stations_df = stations_df.drop_duplicates(subset=['year', 'latitude', 'longitude'])

print(f"\nfound {len(stations_df)} station records for 2026 across the UK.")
display(stations_df.head(10))

Starting historical extraction for 2026 from OpenStreetMap...

--- Fetching Data for 2026 ---
  -> Querying Scotland...
  -> Querying North England...
  -> Querying Wales & West Midlands...
  -> Querying East Midlands & East Anglia...
  -> Querying South England...
  -> Querying Northern Ireland...

Success! Found 1959 station records for 2026 across the UK.


,year,latitude,longitude,name
0,2026,55.750002,-4.936241,Unknown Station
1,2026,55.860672,-4.027778,Coatbridge Police station
2,2026,55.799724,-4.390692,Barrhead Police Station
3,2026,56.036071,-5.431863,Unknown Station
4,2026,56.467454,-2.877516,Broughty Ferry Police Station
5,2026,57.583259,-4.128058,Unknown Station
6,2026,56.481861,-3.450400,Unknown Station
7,2026,54.971733,-2.461151,Haltwhistle Police House
8,2026,55.778664,-2.345754,Duns Police Office
9,2026,57.513373,-4.456678,Muir of Ord Police Station and Council Service...


In [ ]:
import geopandas as gpd
import sqlite3


lsoa_map = gpd.read_file('../data/lsoa_spatial.geojson') 

#conveting station data to spatial format
stations_gdf = gpd.GeoDataFrame(
    stations_df, 
    geometry=gpd.points_from_xy(stations_df.longitude, stations_df.latitude),
    crs="EPSG:4326"
)

#make sure coord system matches
if stations_gdf.crs != lsoa_map.crs:
    stations_gdf = stations_gdf.to_crs(lsoa_map.crs)

#spatial join
stations_in_uk = gpd.sjoin(stations_gdf, lsoa_map, how="inner", predicate="within")

# count them up
lsoa_counts = stations_in_uk.groupby('LSOA21CD').size().reset_index(name='police_station_count')

#save to db
conn = sqlite3.connect('../data/wales_data.db')
lsoa_counts.to_sql('lsoa_infrastructure', conn, if_exists='replace', index=False)
conn.close()

Loading LSOA boundaries...
Converting station data to spatial format...
Filtering stations through the LSOA map...

Cleaned! We went from 1959 raw points down to 1481 UK-verified stations.
Number of LSOAs with at least one station: 1372

Success! The cleaned 'police_station_count' is now in your database.


In [ ]:
import sqlite3

conn = sqlite3.connect('../data/wales_data.db')
#change lsoa21 to lsoa_code
conn.execute('ALTER TABLE lsoa_infrastructure RENAME COLUMN LSOA21CD TO lsoa_code')
conn.commit()
conn.close()

Done! LSOA21CD renamed to lsoa_code in lsoa_infrastructure.


In [ ]:
#remove english lsoas
conn = sqlite3.connect('../data/wales_data.db')
query = '''
SELECT *
FROM lsoa_infrastructure
WHERE lsoa_code LIKE 'W%'
'''

df = pd.read_sql(query, conn)
print(df)

df.to_sql('lsoa_infrastructure', conn, if_exists='replace', index=False)
conn.close()


     lsoa_code  police_station_count
0    W01000051                     2
1    W01000056                     1
2    W01000067                     1
3    W01000080                     1
4    W01000095                     1
..         ...                   ...
152  W01001977                     1
153  W01001990                     1
154  W01002004                     1
155  W01002014                     1
156  W01002019                     1

[157 rows x 2 columns]
